# Build a Debugging Tutor with Gradio

### Learning Objective
In this notebook you will build an AI debugging tutor to learn how system prompts, examples, and settings change the way a language model behaves.

### What You Will Do
- **Edit the system prompt** to change the AI's rules and personality.
- **Toggle few-shot examples** on or off to see how the AI learns from them.
- **Adjust temperature** to control randomness.
- **Try Chain-of-Thought (CoT)** to make the model reason step-by-step before answering.
- **Compare a small local model with a large API model** (optional) to see how size affects quality.
- **Inspect the White Box** to see the exact tokens sent to the model.
- **Keep an experiment log** to compare runs.

### How These Skills Transfer
The workflow you learn here (system prompt, few-shot examples, CoT, context, evaluation) is the same for any LLM application: customer support bots, writing assistants, code review tools, and more.

## 1. Setup

First we import every library the notebook needs. All imports are in this one cell.

**Local inference** means running a model on your own machine. It costs nothing per query but is slower and limited by your hardware.

In [15]:
import gradio as gr # gradio: Build interactive web UIs for machine learning models
from llama_cpp import Llama # llama_cpp: Run GGUF language model files locally on CPU or GPU
import json # json: Read and write structured datas
import time # time: Measure how long each model call takes
import re # re: Search for patterns inside text (used to find <thinking> tags)
# Error: %pip -q install gradio llama-cpp-python

Now we load the model file into memory. This step takes a few seconds.

`context_window_size` sets the maximum number of **tokens** the model can read at once. A token is a small piece of text, roughly 3-4 characters. A bigger window lets the model see more text but uses more memory.

In [16]:
# Change this to match where your model file is located
model_file_path = "/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf"

# Change this to increase or decrease context window
context_window_size = 1024

language_model = Llama(
    model_path=model_file_path,
    n_ctx=context_window_size,
    n_threads=4,          # number of CPU threads to use
    verbose=False,
)

print("Local model loaded. Context window:", context_window_size, "tokens.")

llama_context: n_ctx_per_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Local model loaded. Context window: 1024 tokens.


## 1b. (Optional) OpenAI API Model

The cell below connects to the OpenAI API so you can compare the small local model with a large cloud model.

**Everything is commented out by default.** To enable it:
1. Uncomment every line below the first three lines.
2. Check that `env_file_path` points to your `.env` file.
3. Check that `env_variable_name` matches the key name inside your `.env` file.
4. Re-run this cell, then re-run the Gradio cell.

When enabled, the Gradio UI will show a **Model** dropdown (Local vs API).

In [17]:
# This line keeps the notebook working when the API code below is commented out.
# When you enable the API, the last line below will change this to True.
use_api_model = True

#── UNCOMMENT EVERYTHING BELOW TO ENABLE THE OPENAI API MODEL ──
from openai import OpenAI
from dotenv import load_dotenv
import os
# Error: %pip install openai python-dotenv


# Change this to match where your .env file is located
env_file_path = "api.env"
# Change this to match the variable name INSIDE your .env file (OPENAI_API_KEY=sk-....)
# Change "OPENAI_API_KEY" to OPENAI_API_KEY if OPENAI_API_KEY="sk-...."
env_variable_name = "OPENAI_API_KEY"
# Load OPEN_API_KEY from one env file.
load_dotenv(env_file_path)
openai_api_key = os.getenv(env_variable_name)
print("API Key:", "loaded" if openai_api_key else "NOT FOUND -- check env_file_path and env_variable_name")

if openai_api_key:
    openai_client = OpenAI(api_key=openai_api_key)
    # Change this to try different OpenAI models
    openai_model_name = "gpt-4o-mini"
    use_api_model = True
    print("OpenAI API model enabled:", openai_model_name)
else:
    print("API model NOT enabled.")

API Key: loaded
OpenAI API model enabled: gpt-4o-mini


## 2. System Prompt

**Prompt engineering** is the skill of writing instructions that control how an LLM behaves.

The system prompt defines three things:
- **Role**: What is the AI pretending to be?
- **Rules**: What must it do? What must it never do?
- **Format**: How should the output look?

Both prompts are editable in the Gradio UI:
- **System Prompt**: The base instructions. Always used.
- **CoT Addition**: Extra instructions added when CoT is on. Try changing the thinking steps!

### Experiments to try
- Delete the "NEVER give the corrected code" rule. Does the model start giving answers?
- Edit the CoT addition: change the 4 thinking steps to just 1. Does the output get worse?
- Change "TA" to "senior engineer". Does the tone change?

In [18]:
# This is the prompt students can edit in the Gradio UI.
system_prompt = """
You are a TA helping a student debug their code.
You must NEVER give the corrected code or the direct answer.
Guide the student to find the fix themselves.

Rules:
- Max 25 words per section.
- Do NOT fix the student's code directly.
- If the error is unclear, ask ONE question in Check.

Output format (use these exact headers):

Diagnosis: What symptom the error message describes.
Root cause: Why the code produces that symptom, without revealing the fix.
Check: One step or short example the student can try to verify the cause.
Review: One general principle to prevent this type of bug.
""".strip()


# When CoT is ON, this text is added AFTER the system prompt.
# Students can also edit this in the Gradio UI.
cot_addition = """
IMPORTANT: Before answering, think step-by-step inside <thinking> tags.
In your thinking:
1. What does the error message tell us?
2. What is the student code trying to do?
3. Where exactly is the mismatch between intent and code?
4. What hint can I give WITHOUT revealing the answer?

After </thinking>, write your answer using the headers above.
Keep the answer sections under 25 words each.
""".strip()


print(f" System prompt: {len(system_prompt)}, chars, ~{len(system_prompt)//4} tokens")
print(f" CoT addition: {len(cot_addition)}, chars, ~{len(cot_addition)//4} tokens (added automatically when CoT is on)")

 System prompt: 591, chars, ~147 tokens
 CoT addition: 392, chars, ~98 tokens (added automatically when CoT is on)


## 3. Few-Shot Examples

**In-context learning** means teaching a model by showing it examples inside the prompt. The model sees these examples and mimics the pattern. No retraining is needed.

The tradeoff: each example uses tokens from the context window. The two examples below use about 250 tokens that are then not available for the student question.

Experiment: Toggle few-shot off in the UI. Does the model still follow the four-header format? Small models usually need few-shot more than large models.

In [19]:
# Two examples covering different error types.
# Example 1: TypeError (type mismatch)
# Example 2: Mutable default argument (logic error, no crash)

few_shot_examples = [
    {
        "role": "user",
        "content": "Student code:\nresult = 'Total: ' + 42\n\n"
                   "Error:\nTypeError: can only concatenate str (not \"int\") to str",
    },
    {
        "role": "assistant",
        "content": "Diagnosis: Python cannot add a string and an integer together.\n"
                   "Root cause: The + operator needs both sides to be the same type.\n"
                   "Check: Try str(42) or use an f-string: f'Total: {42}'\n"
                   "Review: When combining text and numbers, convert the number to a string first.",
    },
    {
        "role": "user",
        "content": "Student code:\ndef register(name, attendees=[]):\n"
                   "    attendees.append(name)\n"
                   "    return attendees\n\n"
                   "print(register('Alice'))\n"
                   "print(register('Bob'))  # Returns ['Alice', 'Bob'] not ['Bob']\n\n"
                   "No crash, but Bob's list includes Alice.",
    },
    {
        "role": "assistant",
        "content": "Diagnosis: The list grows across calls instead of resetting.\n"
                   "Root cause: Python creates the default list once, then reuses it every call.\n"
                   "Check: Call register('X') three times. Print id(attendees) each time.\n"
                   "Review: Never use a mutable object (list, dict) as a default argument.",
    },
]

print("Few-shot examples loaded:", len(few_shot_examples), "messages")

Few-shot examples loaded: 4 messages


## 4. Test Scenarios

**Evaluation design**: You cannot improve what you cannot measure.

These five scenarios are our test suite, ordered from easy to hard. Every scenario comes from real bugs that beginners actually hit (sources in the code comments).

Each scenario has optional context layers you can toggle in the UI:
- `spec`: What the assignment asks.
- `docs`: Library documentation.
- `hint`: A worked example showing the correct pattern.

Experiment: **more context uses more tokens but might improve quality. When is it worth it?**

In [20]:
scenarios = {
    "1. TypeError: str + int [Easy]": {
        # Source: stackoverflow.com/questions/1893507 (3M+ views)
        "code": 'age = 25\nmessage = "I am " + age + " years old."\nprint(message)',
        "error": 'TypeError: can only concatenate str (not "int") to str',
        "spec": "Build a greeting string that includes the user's name and age.",
        "docs": "str(x): Convert x to string. f-strings: f'text {var}' embed variables.",
        "hint": "# Two ways to mix strings and numbers:\n"
               "# greeting = 'Score: ' + str(100)\n"
               "# greeting = f'Score: {100}'",
    },
    "2. IndexError: list out of range [Easy]": {
        # Source: stackoverflow.com/questions/1098643
        "code": "fruits = ['apple', 'banana', 'cherry']\n"
               "for i in range(len(fruits)):\n"
               "    print(fruits[i], fruits[i+1])",
        "error": "IndexError: list index out of range",
        "spec": "Print each fruit and the fruit that comes after it.",
        "docs": "len(lst): Number of elements. Last valid index is len(lst)-1.",
        "hint": "# To access pairs, stop one element early:\n"
               "# for i in range(len(lst) - 1):\n"
               "#     print(lst[i], lst[i+1])",
    },
    "3. Mutable default argument [Medium]": {
        # Source: toptal.com - listed as the #1 Python mistake
        "code": "def add_item(item, shopping_list=[]):\n"
               "    shopping_list.append(item)\n"
               "    return shopping_list\n\n"
               "print(add_item('milk'))\n"
               "print(add_item('bread'))  # Expected ['bread'], got ['milk', 'bread']",
        "error": "No crash, but wrong output: the list accumulates across calls.",
        "spec": "Write a function that creates a new shopping list each time.",
        "docs": "Default values are evaluated ONCE when the function is defined, not each call.",
        "hint": "# Safe pattern for mutable defaults:\n"
               "# def func(item, lst=None):\n"
               "#     if lst is None:\n"
               "#         lst = []\n"
               "#     lst.append(item)\n"
               "#     return lst",
    },
    "4. pandas KeyError: column name [Medium]": {
        # Source: stackoverflow.com/questions/17431924 (1M+ views)
        "code": "import pandas as pd\n"
               "student_data = pd.DataFrame({'Name': ['Alice', 'Bob'], 'Age': [25, 30]})\n"
               "avg_age = student_data['age'].mean()",
        "error": "KeyError: 'age'",
        "spec": "Calculate the average age from the DataFrame.",
        "docs": "Column access is case-sensitive: df['Age'] != df['age']. Use df.columns.tolist() to check.",
        "hint": "# Always check column names first:\n"
               "# print(student_data.columns.tolist())\n"
               "# Then use exact case: student_data['Age'].mean()",
    },
    "5. UnboundLocalError: variable scope [Hard]": {
        # Source: toptal.com - listed as Python mistake #4
        "code": "count = 0\n\n"
               "def increment():\n"
               "    count += 1\n"
               "    return count\n\n"
               "print(increment())",
        "error": "UnboundLocalError: cannot access local variable 'count'",
        "spec": "Write a function that increments a global counter by 1.",
        "docs": "Assignment inside a function makes the variable local. Use 'global' to modify a global variable.",
        "hint": "# Two ways to fix:\n"
               "# 1. global keyword: global count\n"
               "# 2. Better: pass as argument:\n"
               "#    def increment(current):\n"
               "#        return current + 1",
    },
}

print("Loaded", len(scenarios), "scenarios.")

Loaded 5 scenarios.


## 5. Inference Engine

This code assembles all the pieces (system prompt + few-shot + user input + context) into a single prompt, sends it to the model, and records what happens.

When **Chain-of-Thought** is enabled, the model is asked to reason inside `<thinking>` tags before answering. The code separates the thinking from the final answer:
- The **chat panel** shows only the final answer.
- The **Thinking tab** shows the reasoning process.

Note: Small local models often produce broken or messy thinking. That is expected and is itself an important observation. Large API models handle CoT much better.

**Why functions?** Gradio needs a function to call when you click a button. That is the only reason this section uses `def`.

In [21]:
experiment_log = []

def format_log():
    """Build a readable text block showing all past runs."""
    if len(experiment_log) == 0:
        return "No experiments yet. Click Ask to start."
    
    log_lines = []
    for record in reversed(experiment_log[-20:]):
        header = "Run #" + str(record["run"]) + " | " + record["scenario"] + " | " + record["mode"] + " | " + str(record["latency"]) + "s"
        log_lines.append(header)
        log_lines.append("------------------------------------------------------------")
        log_lines.append(record["response"])
        log_lines.append("=" * 60 + "\n")
    return "\n".join(log_lines)


def run_tutor(user_text, system_prompt_text, cot_addition_text, use_few_shot,
              temperature, max_tokens, scenario_name, use_spec, use_docs,
              use_hint, use_cot, model_choice, chat_history):
    """Called every time the student clicks Ask."""
    
    # ── Step 1: Build the system prompt ──
    # Always start with whatever the student typed in the textbox.
    # If CoT is on, append the CoT instructions after it.
    active_prompt = system_prompt_text
    if use_cot == True:
        active_prompt = active_prompt + "\n\n" + cot_addition_text

    messages = []
    messages.append({"role": "system", "content": active_prompt})

    # Add few-shot examples if the checkbox is on
    if use_few_shot == True:
        for example in few_shot_examples:
            messages.append(example)

    # ── Step 2: Build the user message ──
    user_message = ""

    if scenario_name != "(none)" and scenario_name in scenarios:
        scenario = scenarios[scenario_name]
        user_message = "Student code:\n" + scenario["code"] + "\n\n"
        user_message = user_message + "Error:\n" + scenario["error"] + "\n"

        if use_spec == True and scenario.get("spec", "") != "":
            user_message = user_message + "\nAssignment spec:\n" + scenario["spec"] + "\n"
        if use_docs == True and scenario.get("docs", "") != "":
            user_message = user_message + "\nDocs reference:\n" + scenario["docs"] + "\n"
        if use_hint == True and scenario.get("hint", "") != "":
            user_message = user_message + "\nWorked example:\n" + scenario["hint"] + "\n"

        if user_text.strip() != "":
            user_message = user_message + "\nStudent note: " + user_text.strip()
    else:
        user_message = user_text.strip()
        if user_message == "":
            return chat_history, "", "Type a question or select a scenario.", format_log(), "", ""

    # If CoT is on, remind the model to start with <thinking>
    if use_cot == True:
        user_message = user_message + "\n\nRemember: start your response with <thinking>"

    messages.append({"role": "user", "content": user_message})

    # ── Step 3: White Box ──
    white_box_lines = []
    for msg in messages:
        white_box_lines.append("[" + msg["role"].upper() + "]")
        white_box_lines.append(msg["content"])
        white_box_lines.append("----------------------------------------\n")
    white_box_text = "\n".join(white_box_lines)

    # ── Step 4: Count input tokens ──
    input_tokens = len(language_model.tokenize(white_box_text.encode("utf-8")))

    # CoT needs extra output tokens for the thinking block
    effective_max_tokens = int(max_tokens)
    if use_cot == True:
        effective_max_tokens = min(int(max_tokens) + 200, 512)

    # ── Step 5: Call the model ──
    start_time = time.perf_counter()

    is_api = (model_choice == "API" and use_api_model == True)

    if is_api:
        api_response = openai_client.chat.completions.create(
            model=openai_model_name,
            messages=messages,
            max_tokens=effective_max_tokens,
            temperature=float(temperature),
        )
        raw_response = api_response.choices[0].message.content.strip()
        output_tokens = api_response.usage.completion_tokens
        mode_label = "API"
    else:
        local_response = language_model.create_chat_completion(
            messages=messages,
            max_tokens=effective_max_tokens,
            temperature=float(temperature),
            top_p=1.0,
        )
        raw_response = local_response["choices"][0]["message"]["content"].strip()
        usage = local_response.get("usage", {})
        output_tokens = usage.get("completion_tokens", 0)
        mode_label = "Local"

    latency = round(time.perf_counter() - start_time, 2)

    # ── Step 6: Separate thinking from answer ──
    thinking_text = ""
    display_answer = raw_response

    if "<thinking>" in raw_response:
        match = re.search(r"<thinking>(.*?)</thinking>", raw_response, re.DOTALL)
        if match:
            thinking_text = match.group(1).strip()
            display_answer = re.sub(r"<thinking>.*?</thinking>", "", raw_response, flags=re.DOTALL).strip()
        else:
            # Model started <thinking> but never closed it — show everything as thinking
            thinking_text = raw_response.replace("<thinking>", "").strip()
            display_answer = "(Model started thinking but did not produce a final answer. Try turning CoT off, or use the API model.)"

    if use_cot == True:
        mode_label = mode_label + " + CoT"

    # ── Step 7: Metrics ──
    pct = (input_tokens * 100) // context_window_size
    metrics = "Model: " + mode_label + "\n"
    metrics = metrics + "Input tokens: " + str(input_tokens) + " / " + str(context_window_size) + " (" + str(pct) + "%)\n"
    metrics = metrics + "Output tokens: " + str(output_tokens) + "\n"
    metrics = metrics + "Latency: " + str(latency) + " seconds"

    # ── Step 8: Experiment log ──
    experiment_log.append({
        "run": len(experiment_log) + 1,
        "scenario": scenario_name[:25],
        "mode": mode_label,
        "latency": latency,
        "response": display_answer,
    })

    # ── Step 9: Chat history with token badge ──
    badge = "\n`" + str(latency) + "s | in:" + str(input_tokens) + " | out:" + str(output_tokens) + " | " + mode_label + "`"

    if chat_history == None:
        chat_history = []
    
    if user_text.strip() != "":
        display_name = user_text.strip()
    else:
        display_name = "[" + scenario_name + "]"

    chat_history.append({"role": "user", "content": display_name})
    chat_history.append({"role": "assistant", "content": display_answer + badge})

    # ── Step 10: Thinking tab text ──
    if use_cot == False:
        thinking_display = "(CoT is off. Check the CoT box to see the model think step-by-step.)"
    elif thinking_text == "":
        thinking_display = "(CoT is on, but the model did not produce <thinking> tags. Small models sometimes ignore CoT instructions.)"
    else:
        thinking_display = thinking_text

    return chat_history, white_box_text, metrics, format_log(), "", thinking_display


print("Inference engine ready.")

Inference engine ready.


## 6. Gradio Chatbot UI

This is where everything comes together. The UI has four tabs at the bottom:
1. **White Box**: The exact prompt sent to the model.
2. **Thinking**: The model's reasoning process (only when CoT is on).
3. **Metrics**: Token counts and latency for the current run.
4. **Experiment Log**: Every run this session, so you can compare.

### Experiment Guide

| # | Experiment | What to change | What to look for |
|---|-----------|---------------|------------------|
| 1 | **Prompt matters** | Delete the "NEVER give the answer" rule | Does the model start giving solutions? |
| 2 | **Few-shot effect** | Toggle few-shot off, run Scenario 1, then on | Does the 4-header format hold? |
| 3 | **Context scaling** | Run Scenario 3: no context, +spec, +all | More tokens — better response? |
| 4 | **Temperature** | Same scenario at 0.0, 0.5, 1.0 | Run 2-3x each. More random at high temp? |
| 5 | **CoT on small model** | Check CoT, run Scenario 5 on Local | Does the structure break? Check the Thinking tab. |
| 6 | **CoT on large model** | Check CoT, run Scenario 5 on API | Does reasoning improve? Compare Thinking tabs. |
| 7 | **Compare the log** | After 6+ runs, check Experiment Log | Which settings gave best quality vs speed? |

In [22]:
scenario_choices = ["(none)"]
for name in scenarios:
    scenario_choices.append(name)

model_choices = ["Local"]
if use_api_model == True:
    model_choices.append("API")

custom_css = """
    .gradio-container { font-family: sans-serif !important; }
    code, pre, textarea { font-family: monospace !important; }
    .chatbot .message code { background: #f1f5f9 !important; padding: 2px 4px !important; }
    .thinking-box textarea { background: #fffbeb !important; }
"""

app = gr.Blocks(title="Debugging Tutor")

with app:
    gr.Markdown("# Debugging Tutor\nExperiment with every part of the LLM pipeline. Change settings, then click **Ask**.")

    with gr.Row():
        # ── LEFT: Chat ──
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Debugging Tutor", height=400)
            with gr.Row():
                user_input = gr.Textbox(label="Your message", scale=10, lines=3)
                ask_btn = gr.Button("Ask", variant="primary", scale=1, min_width=80)
            clear_btn = gr.Button("New Chat", variant="secondary")

        # ── RIGHT: Controls ──
        with gr.Column(scale=2):
            gr.Markdown("### Controls")
            scenario_dd = gr.Dropdown(choices=scenario_choices, value="(none)", label="Scenario")
            model_dd = gr.Dropdown(choices=model_choices, value="Local", label="Model")

            with gr.Accordion("Prompting Strategy", open=True):
                cot_cb = gr.Checkbox(value=False, label="Enable Chain-of-Thought (CoT)")
                few_shot_cb = gr.Checkbox(value=True, label="Enable few-shot examples")

            with gr.Accordion("Context Layers", open=False):
                spec_cb = gr.Checkbox(value=False, label="spec (assignment description)")
                docs_cb = gr.Checkbox(value=False, label="docs (library reference)")
                hint_cb = gr.Checkbox(value=False, label="hint (worked example)")

            with gr.Accordion("Model Parameters", open=False):
                temp_slider = gr.Slider(minimum=0.0, maximum=1.5, value=0.2, step=0.1, label="Temperature")
                max_tok_slider = gr.Slider(minimum=64, maximum=512, value=192, step=32, label="Max output tokens")

    # ── FULL-WIDTH: Editable prompts ──
    with gr.Accordion("System Prompt & CoT (click to edit)", open=False):
        with gr.Row():
            system_prompt_box = gr.Textbox(value=system_prompt, label="System Prompt (always used)", lines=6, interactive=True, scale=3)
            cot_addition_box = gr.Textbox(value=cot_addition, label="CoT Addition (added when CoT is on)", lines=6, interactive=True, scale=2)
        reset_prompt_btn = gr.Button("Reset both to default", variant="secondary", size="sm")

    # ── FULL-WIDTH: Inspection tabs ──
    with gr.Tabs():
        with gr.TabItem("White Box"):
            white_box = gr.Textbox(label="Full prompt sent to model", lines=15, interactive=False)
        with gr.TabItem("Thinking"):
            thinking_box = gr.Textbox(
                label="Model reasoning (only when CoT is on)",
                lines=12, interactive=False,
                elem_classes=["thinking-box"],
            )
        with gr.TabItem("Metrics"):
            metrics_box = gr.Textbox(label="Token usage and latency", lines=6, interactive=False)
        with gr.TabItem("Experiment Log"):
            log_box = gr.Textbox(label="All runs this session", lines=12, interactive=False, value=format_log())

    # ── Wire up buttons ──
    inputs = [
        user_input, system_prompt_box, cot_addition_box, few_shot_cb, temp_slider,
        max_tok_slider, scenario_dd, spec_cb, docs_cb, hint_cb,
        cot_cb, model_dd, chatbot
    ]
    outputs = [chatbot, white_box, metrics_box, log_box, user_input, thinking_box]

    ask_btn.click(fn=run_tutor, inputs=inputs, outputs=outputs)
    user_input.submit(fn=run_tutor, inputs=inputs, outputs=outputs)

    def clear_chat():
        return [], "", "", format_log(), "", ""

    clear_btn.click(fn=clear_chat, outputs=[chatbot, white_box, metrics_box, log_box, user_input, thinking_box])

    # Reset BOTH prompts to their defaults
    def reset_prompts():
        return system_prompt, cot_addition

    reset_prompt_btn.click(fn=reset_prompts, outputs=[system_prompt_box, cot_addition_box])

app.launch(share=True, inline=True, height=900, theme=gr.themes.Soft(), css=custom_css)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://3c655bf6df35f8f187.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Reflection

After running at least 6-8 experiments, answer these questions:

### Prompt Engineering
1. You edited the system prompt live. What rule had the biggest impact? What happened when you removed it?
2. Did few-shot examples help more with output *format* or *content*? How can you tell from the White Box?

### Chain-of-Thought
3. Turn on CoT and run Scenario 5 on the **local** model. Look at the Thinking tab. Is the reasoning coherent, or does the structure break down? Why might a 1.5B-parameter model struggle with CoT?
4. Now run the same scenario with CoT on the **API** model (if enabled). Compare the Thinking tab output. What differences do you notice?
5. Try CoT on Scenario 1 (easy). Was the thinking useful, or just wasted tokens?

### Context Window
6. How did input tokens change as you added context layers (spec, docs, hint)? Was there a sweet spot?

### Temperature
7. Run the same scenario 3 times at temperature 0.0, then 3 times at 1.0. How consistent are the responses?

### The Bigger Picture
8. You now have several prompting tools: system prompt editing, few-shot, CoT, context layers, temperature, and model choice. If you were building a production debugging tutor, which combination would you pick and why?
9. The workflow you practiced applies to any LLM application. Pick a different use case and sketch what each component would look like.